In [3]:
import spacy
import random
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import os

# Cargar las variables de entorno desde el archivo .env
load_dotenv()

# Instalar el modelo en español de spaCy
!python -m spacy download es_core_news_sm

nlp = spacy.load("es_core_news_sm")

# Configuración de la API de TMDb
API_KEY = os.getenv("TMDB_API_KEY")  # Obtener la API Key desde el archivo .env
BASE_URL = "https://api.themoviedb.org/3"

def extract_movie_data(movie_id: int) -> dict:
    """
    Extrae los datos de una película desde la API de TMDb.
    """
    url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&language=es-ES"
    response = requests.get(url)
    if response.status_code == 200:
        movie_data = response.json()
        
        # Extraer reseñas
        reviews_url = f"{BASE_URL}/movie/{movie_id}/reviews?api_key={API_KEY}&language=es-ES"
        reviews_response = requests.get(reviews_url)
        reviews = []
        if reviews_response.status_code == 200:
            reviews_data = reviews_response.json()
            reviews = [review['content'] for review in reviews_data['results']]  # Extraemos el contenido de las reseñas
        
        movie_data['reviews'] = reviews
        return movie_data
    else:
        print(f"Error al obtener la película {movie_id}: {response.status_code}")
        return {}

def transform_movie_data(raw_data: dict) -> dict:
    """
    Transforma y limpia los datos crudos extraídos:
    - Tokeniza la sinopsis.
    - Convierte la fecha de estreno a un objeto datetime.
    - Extrae los géneros en formato de lista.
    """
    if not raw_data:
        return {}
    
    # Tokenización de la sinopsis (overview)
    overview = raw_data.get("overview", "")
    doc = nlp(overview)
    tokens = [token.text for token in doc]

    # Conversión de fecha de estreno
    release_date_str = raw_data.get("release_date", "")
    try:
        release_date = datetime.strptime(release_date_str, "%Y-%m-%d").date() if release_date_str else None
    except Exception as e:
        release_date = None

    transformed = {
        "id": raw_data.get("id"),
        "titulo": raw_data.get("title"),
        "sinopsis": overview,
        "tokens_sinopsis": tokens,
        "fecha_estreno": release_date,
        "puntuacion": raw_data.get("vote_average"),
        "numero_votos": raw_data.get("vote_count"),
        "generos": [genre["name"] for genre in raw_data.get("genres", [])],
        "idioma_original": raw_data.get("original_language"),
        "idiomas_disponibles": [lang["name"] for lang in raw_data.get("spoken_languages", [])],
        "reseñas": raw_data.get("reviews", [])  # Reseñas extraídas
    }
    return transformed

def load_data_to_parquet(data_list: list, file_name: str = "C:/Users/jugas/OneDrive/Escritorio/Movie recommender/MVP_sistema_recomendacion/proyecto/data/movies.parquet"):
    """
    Crea un DataFrame con los datos transformados y lo guarda en un archivo Parquet.
    """
    df = pd.DataFrame(data_list)
    df.to_parquet(file_name, index=False)
    print(f"Datos cargados en {file_name}")

def get_random_movie_ids(total_ids=300):
    """
    Obtiene 300 IDs de películas de forma aleatoria desde la API de TMDb.
    """
    movie_ids = []
    page = 1
    
    # Mientras no tengamos 300 IDs de películas
    while len(movie_ids) < total_ids:
        url = f"{BASE_URL}/discover/movie?api_key={API_KEY}&language=es-ES&page={page}"
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            movie_ids.extend([movie['id'] for movie in data['results']])
            page += 1  # Aumentar la página para obtener más resultados
        else:
            print(f"Error al obtener las películas de la página {page}: {response.status_code}")
            break

    # Seleccionamos los primeros 300 IDs aleatorios
    return random.sample(movie_ids, total_ids)[:total_ids]

def main():
    # Obtener 300 IDs de películas aleatorios
    movie_ids = get_random_movie_ids(300)
    transformed_data = []

    for movie_id in movie_ids:
        raw_data = extract_movie_data(movie_id)
        movie_data = transform_movie_data(raw_data)
        if movie_data:
            transformed_data.append(movie_data)

    if transformed_data:
        load_data_to_parquet(transformed_data)
    else:
        print("No se extrajeron datos.")

if __name__ == "__main__":
    main()



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.9 MB 10.1 MB/s eta 0:00:02
     ------------ --------------------------- 3.9/12.9 MB 12.4 MB/s eta 0:00:01
     --------------------- ------------------ 7.1/12.9 MB 13.6 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.9 MB 14.5 MB/s eta 0:00:01
     --------------------------------------- 12.9/12.9 MB 13.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
Datos cargados en C:/Users/jugas/OneDrive/Escritorio/Movie recommender/MVP_sistema_recomendacion/proyecto/data/movies.parquet
